# 03 · Evaluation & Model Comparison

Qualitative demos are persuasive; numbers are decisive. This notebook measures
the model on a validation slice and compares the **base backbone (zero-shot)**
against a **LoRA fine-tune**, then visualises the difference.

1. Evaluate the base model on a small validation subset (`evaluate_model`).
2. Attach the LoRA adapter (if configured) and re-evaluate on the *same* pages.
3. Run `research.compare_models` for a head-to-head on the test split and render
   the comparison table + bar chart.
4. Plot the confidence distribution and (if a training run exists) loss curves.

> **Requirements.** Needs the full dependencies plus `evaluate`/`nltk`-style
> metric libs (they degrade gracefully to `0.0` when missing). If no
> `config.inference.adapter_path` is set, the fine-tuned column is `None` and you
> get a base-only baseline — still a useful artifact, and it will not crash.
> Keep `max_samples` small: generation is the bottleneck.


## Setup

In [ ]:
# --- Make the repo root importable (works whether run from notebooks/ or root)
import sys
from pathlib import Path

# Walk up until we find the repo marker (pyproject.toml); fall back to parent.
_here = Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists()),
    _here.parent,
)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Repo root on sys.path: {_root}")


In [ ]:
from pathlib import Path
from configs import load_config
from utils.logging_utils import get_logger

logger = get_logger("nb.evaluation")

config = load_config()
# Keep the eval slice tiny in the notebook — VLM generation dominates runtime.
MAX_SAMPLES = 16

print("Dataset      :", config.data.dataset_name)
print("Backbone     :", config.model.model_id)
print("Adapter path :", config.inference.adapter_path or "(none — base-only run)")
print("Eval samples :", MAX_SAMPLES)


## Evaluate the base model

`evaluate_model` runs generation over the samples, computes the aggregate QA
metrics (exact match, ANLS, token-F1, BLEU/ROUGE, mean confidence), profiles
latency/throughput/peak memory, and returns per-sample predictions — one stable
dict consumed by the report builder and the plots below.


In [ ]:
from preprocessing.datasets import load_samples
from evaluation.evaluate import evaluate_model
from models import build_model

# Shared, fixed slice so base and fine-tuned are scored on identical pages.
eval_samples = load_samples(config, config.data.val_split or "validation", MAX_SAMPLES)
print(f"Evaluating on {len(eval_samples)} validation samples...\n")

model = build_model(config)  # base backbone on-device, no adapter attached yet
base_result = evaluate_model(model, eval_samples, config, profile=True)

print("BASE metrics")
for key, value in base_result["metrics"].items():
    print(f"  {key:>15}: {value:.4f}" if isinstance(value, (int, float)) else f"  {key:>15}: {value}")
print("\nBASE profile:", base_result["profile"])


## Attach the LoRA adapter and re-evaluate

If an adapter is configured we wrap the resident backbone in a `PeftModel` *in place* (no second multi-GB copy) and score the same pages. Without one, we skip this and compare against the base baseline only.

In [ ]:
adapter_path = config.inference.adapter_path
finetuned_result = None

if adapter_path and Path(adapter_path).exists():
    print(f"Attaching adapter from {adapter_path} and re-evaluating...\n")
    model.load_adapter(adapter_path)            # in-place LoRA wrap
    finetuned_result = evaluate_model(model, eval_samples, config, profile=True)

    print("FINE-TUNED metrics")
    for key, value in finetuned_result["metrics"].items():
        print(f"  {key:>15}: {value:.4f}" if isinstance(value, (int, float)) else f"  {key:>15}: {value}")
else:
    print("No usable adapter_path — showing the base (zero-shot) baseline only.")
    print("Train an adapter (training/train.py) and set config.inference.adapter_path"
          " to populate the fine-tuned column.")


In [ ]:
# Side-by-side headline metrics (base vs fine-tuned, when available).
import pandas as pd

headline = ["exact_match", "anls", "token_f1", "avg_confidence"]
rows = []
for metric in headline:
    base_v = base_result["metrics"].get(metric)
    ft_v = finetuned_result["metrics"].get(metric) if finetuned_result else None
    rows.append({
        "Metric": metric,
        "Base": base_v,
        "Fine-tuned": ft_v,
        "Delta": (ft_v - base_v) if (ft_v is not None and base_v is not None) else None,
    })

pd.DataFrame(rows).set_index("Metric")


## Head-to-head via `research.compare_models`

`compare_models` is the reusable driver behind the CLI report: it loads the test
split once, evaluates base and (if present) fine-tuned on the *same* documents,
computes deltas, and pulls training time from `trainer_state.json` when a run
exists. `build_comparison_dataframe` turns that into a tidy table, and
`plot_model_comparison` renders the grouped bar chart the report embeds.


In [ ]:
from research.compare import compare_models, build_comparison_dataframe

# Same tiny cap so this stays notebook-fast; it evaluates the *test* split.
comparison = compare_models(config, max_samples=MAX_SAMPLES)

df = build_comparison_dataframe(comparison)
df


In [ ]:
from pathlib import Path
from visualization.plots import (
    set_style,
    plot_model_comparison,
    plot_confidence_distribution,
)

set_style()  # consistent house style across every figure
reports_dir = Path(config.report_dir)
reports_dir.mkdir(parents=True, exist_ok=True)

# Grouped bar: base vs fine-tuned on the quality metrics (all on a [0,1] scale).
bar_path = plot_model_comparison(df, reports_dir / "nb_model_comparison.png")
print("Saved:", bar_path)

from IPython.display import Image as IPyImage
IPyImage(filename=str(bar_path))


## Confidence distribution

Per-sample confidences come straight out of the evaluation predictions. A well-behaved fine-tune should be *both* more accurate and better calibrated — worth eyeballing.

In [ ]:
# Use the fine-tuned predictions if we have them, else the base run.
src = finetuned_result or base_result
confidences = [p["confidence"] for p in src["predictions"] if p.get("confidence") is not None]

conf_path = plot_confidence_distribution(confidences, reports_dir / "nb_confidence.png")
print("Saved:", conf_path, f"({len(confidences)} predictions)")
IPyImage(filename=str(conf_path))


## Training curves (optional)

If a fine-tuning run has written a `trainer_state.json`, plot its train/eval loss and the validation metric so the evaluation numbers above have training context. Skipped cleanly when no run is present.

In [ ]:
from visualization.plots import plot_training_curves

# Look for a trainer_state.json under the configured output tree.
output_root = Path(config.output_root)
states = sorted(output_root.rglob("trainer_state.json")) if output_root.exists() else []

if states:
    state = states[-1]  # most recent-ish; any is fine for a demo
    print("Plotting curves from:", state)
    curve_paths = plot_training_curves(str(state), reports_dir)
    for p in curve_paths:
        display(IPyImage(filename=str(p)))
else:
    print("No trainer_state.json found under", output_root,
          "— run training/train.py first to get loss curves.")


## Rendered comparison artifact

The CLI (`python -m research.compare`) writes `reports/comparison.csv` and `reports/comparison.md`. If they exist, surface the CSV here as the notebook's final tabular summary.

In [ ]:
comparison_csv = reports_dir / "comparison.csv"
if comparison_csv.exists():
    display(pd.read_csv(comparison_csv))
else:
    print(f"{comparison_csv} not found. Generate it with:")
    print("    python -m research.compare --max-samples 16")
    print("(this notebook's compare_models call already produced the in-memory table above)")


## Takeaways

- A single tiny slice already tells the story: **ANLS / token-F1** move with the
  LoRA fine-tune while latency and memory stay flat (the adapter adds a sliver of
  parameters). Scale `MAX_SAMPLES` up for a publication-grade number.
- Every figure and table here is the *same* code the CLI report and Streamlit
  dashboard use — the notebook is a live view of the production evaluation path,
  not a parallel re-implementation.
